In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper
import time
import random
from urllib.parse import urljoin
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait


In [7]:
url = "https://fotosource.com/store-selector"

In [ ]:
# Setup Chrome options
options = Options() 
options.add_argument("user-agent=Mozilla/5.0    ") # Replace with your actual user agent
#options.add_argument("--headless")  # Optional: Run without opening the browser
service = Service("C://Users//sarda//Downloads//chromedriver-win64//chromedriver-win64//chromedriver.exe")  # <-- Replace with your path


In [57]:
# Launch browser
driver = webdriver.Chrome(service=service, options=options)

In [58]:
driver.get(url)

In [ ]:
canadian_provinces = [
    "Alberta",
    "British Columbia",
    "Manitoba",
    "New-Brunswick",
    "Newfoundland and Labrador",
    "Nova Scotia",
    "Ontario",
    "Prince Edward Island",
    "Quebec",
    "Saskatchewan"
]

In [60]:
from selenium.webdriver.support import expected_conditions as EC

names = []
address = []
cities = []
emails = []
phones = []
for r in canadian_provinces:
    driver.get(url)
    # Wait until the select element is present
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.NAME, "region"))
    )
    select = Select(driver.find_element(By.NAME, "region"))
    select.select_by_visible_text(r)
    time.sleep(2)  # Wait for the page to load after selection
    
    store_count = driver.find_element(By.CSS_SELECTOR, ".d-store-finder-store-count.d-store-finder-store-count").text.replace(" Stores found", "")
    
    
    #click on all available stores one by one 
    for i in range(int(store_count)):
        store_click = driver.find_elements(By.CSS_SELECTOR, ".d-stores-map-store.d-store-finder-store.d-stores-map-store-visible")
        store_click[i].click()
        time.sleep(2) 
        # Wait for the detailed store information to load
        
        store_name = driver.find_element(By.CSS_SELECTOR, ".d-stores-map-detailed-name.d-store-finder-detailed-name").text
        names.append(store_name)
        store_address = driver.find_element(By.CSS_SELECTOR, ".d-stores-map-detailed-address.d-store-finder-detailed-address").text
        address.append(store_address)
        city_name = driver.find_element(By.CSS_SELECTOR, ".d-stores-map-detailed-city.d-store-finder-detailed-city").text
        cities.append(city_name)
        email = driver.find_element(By.CSS_SELECTOR, ".d-stores-map-detailed-info-column-email-text.d-stores-map-detailed-email.d-store-finder-detailed-email")
        href = email.get_attribute("href")
        if href:
            email = href.replace("mailto:", "").strip()
        emails.append(email)
        phone = driver.find_element(By.CSS_SELECTOR, ".d-stores-map-detailed-phone.d-store-finder-detailed-phone").text
        phones.append(phone)

        #click back and the click on next store
        button = driver.find_element(By.CSS_SELECTOR, ".d-stores-map-detailed-back-text")
        button.click()
        time.sleep(2)  # Wait for the page to load after clicking back

        

# Create a DataFrame to store the results
df = pd.DataFrame({
    
   "Store Name": names,
    "Address": address,
    "City": cities,
    "Email": emails,
    "Phone": phones
})

    



In [62]:
df.to_csv("store_finder_results.csv", index=False)

In [ ]:
data = pd.read_csv("store_finder_results.csv")

In [64]:
data["Region"]= data['City'].str.findall(r'\b[A-Z]{2}\b')

In [65]:
data

,Store Name,Address,City,Email,Phone,Region
0,Shutter Bee Photography - Bonnyville,"5009 50th Ave, PO Box 7118","Bonnyville, AB, Canada",shutterbee.bville@outlook.com,(780) 826-5733,[AB]
1,FastFoto,1217 3800 Memorial Dr. N.E.,"Calgary, AB, Canada",alitahan@hotmail.com,403-454-2474,[AB]
2,Calgary Custom Photo Services - North,1632 14 Ave NW,"Calgary, AB, Canada",ccpsnorth@teamccps.com,403-282-3800,[AB]
3,Calgary Custom Photo Services - South,9737 Macleod Tr SW,"Calgary, AB, Canada",ccpssouth@teamccps.com,4032553880,[AB]
4,McBain Camera Ltd. - Head Office,10805 107 Ave,"Edmonton, AB, Canada",sales.107ave@mcbaincamera.com,1-800-661-6980,[AB]
...,...,...,...,...,...,...
112,Victoriaville Photo Inc. Foto Source,136 Notre-Dame Est,"Victoriaville, QC, Canada",info@pixm.com,819 758-7536,[QC]
113,Wells Camera & Sound Foto Source,1102 Main Street North,"Moose Jaw, SK, Canada",wells.sales@sasktel.net,(306) 693-3494,[SK]
114,Bird Films,"4621 Rae Street, #3","Regina, SK, Canada",birdfilms@accesscomm.ca,(306) 586-0311,[SK]
115,Don's Photo - Regina,210 - 2410 Dewdney Avenue,"Regina, SK, Canada",regina@donsphoto.ca,(306) 347-7887,[SK]


In [66]:
data['Region'] = data['Region'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else '')


In [67]:
data.head()

,Store Name,Address,City,Email,Phone,Region
0,Shutter Bee Photography - Bonnyville,"5009 50th Ave, PO Box 7118","Bonnyville, AB, Canada",shutterbee.bville@outlook.com,(780) 826-5733,AB
1,FastFoto,1217 3800 Memorial Dr. N.E.,"Calgary, AB, Canada",alitahan@hotmail.com,403-454-2474,AB
2,Calgary Custom Photo Services - North,1632 14 Ave NW,"Calgary, AB, Canada",ccpsnorth@teamccps.com,403-282-3800,AB
3,Calgary Custom Photo Services - South,9737 Macleod Tr SW,"Calgary, AB, Canada",ccpssouth@teamccps.com,4032553880,AB
4,McBain Camera Ltd. - Head Office,10805 107 Ave,"Edmonton, AB, Canada",sales.107ave@mcbaincamera.com,1-800-661-6980,AB


In [68]:
data = data[data.columns[[5, 0, 1, 2, 3, 4]]]

In [70]:
data.to_csv("store_finder_results_with_region.csv", index=False)